# 🌍 Climate NER — Self-Contained Colab Notebook

**Just open this file in Colab and hit `Runtime → Run All`.**

Everything is embedded — no Drive mount, no git clone, no zip upload needed.

| | |
|---|---|
| **Base model** | `nasa-impact/nasa-smd-ibm-v0.1` (INDUS / RoBERTa-base) |
| **Dataset** | `ibm-research/Climate-Change-NER` (534 climate abstracts, 27 IOB tags) |
| **Runtime** | T4 GPU → Runtime → Change runtime type → **T4 GPU** |
| **Training time** | ~12–18 min on T4 for 5 epochs |

---
## ✅ Cell 0 — Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️  No GPU detected.')
    print('   Go to: Runtime → Change runtime type → T4 GPU, then re-run.')

---
## 📦 Cell 1 — Install dependencies

In [ ]:
%pip install -q --upgrade pip 'setuptools<82' wheel
%pip install -q transformers datasets accelerate seqeval evaluate huggingface-hub
print('Dependencies installed ✓')

---
## 📝 Cell 2 — Write source files to /content/climate_ner/

All project code is embedded here — no external files needed.

> **Note:** Our evaluation module is named `ner_eval.py` (not `evaluate.py`) to
> avoid shadowing the installed HuggingFace `evaluate` library, which Python
> would find first when resolving imports from the same directory.

In [ ]:
import os
os.makedirs('/content/climate_ner', exist_ok=True)
open('/content/climate_ner/__init__.py', 'w').close()
print('Package directory created ✓')

In [ ]:
%%writefile /content/climate_ner/dataset.py
"""
dataset.py — Parse ibm-research/Climate-Change-NER raw IOB lines
             into token-classification-ready HuggingFace Dataset rows.

Audit decisions baked in:
  Q1: Raw text lines, not pre-parsed -> custom _parse_iob_lines() required.
  Q4: Pre-split at document level by IBM (unique hashes in -DOCSTART- lines)
      -> use train/validation/test splits as-is, no re-splitting needed.
  Q5: O-label dominates (~83% of tokens) -> compute_class_weights() provides
      inverse-frequency weights capped at 10x for weighted cross-entropy loss.
"""
from __future__ import annotations
import collections
from datasets import load_dataset, Dataset, DatasetDict

LABEL_NAMES: list[str] = [
    "O",
    "B-climate-assets",           "I-climate-assets",
    "B-climate-datasets",         "I-climate-datasets",
    "B-climate-greenhouse-gases", "I-climate-greenhouse-gases",
    "B-climate-hazards",          "I-climate-hazards",
    "B-climate-impacts",          "I-climate-impacts",
    "B-climate-mitigations",      "I-climate-mitigations",
    "B-climate-models",           "I-climate-models",
    "B-climate-nature",           "I-climate-nature",
    "B-climate-observations",     "I-climate-observations",
    "B-climate-organisms",        "I-climate-organisms",
    "B-climate-organizations",    "I-climate-organizations",
    "B-climate-problem-origins",  "I-climate-problem-origins",
    "B-climate-properties",       "I-climate-properties",
]

LABEL2ID: dict[str, int] = {lbl: i for i, lbl in enumerate(LABEL_NAMES)}
ID2LABEL: dict[int, str] = {i: lbl for i, lbl in enumerate(LABEL_NAMES)}
HF_DATASET_ID = "ibm-research/Climate-Change-NER"


def _parse_iob_lines(raw_rows: list[dict]) -> list[dict]:
    sentences: list[dict] = []
    current_tokens: list[str] = []
    current_labels: list[int] = []
    for row in raw_rows:
        line: str = row["text"].strip()
        if line.startswith("-DOCSTART-"):
            continue
        if not line:
            if current_tokens:
                sentences.append({"tokens": current_tokens, "ner_tags": current_labels})
                current_tokens, current_labels = [], []
            continue
        parts = line.split()
        if len(parts) < 2:
            continue
        label = parts[-1]
        if label not in LABEL2ID:
            label = "O"
        current_tokens.append(parts[0])
        current_labels.append(LABEL2ID[label])
    if current_tokens:
        sentences.append({"tokens": current_tokens, "ner_tags": current_labels})
    return sentences


def load_climate_ner(hf_dataset_id: str = HF_DATASET_ID) -> DatasetDict:
    raw = load_dataset(hf_dataset_id)
    return DatasetDict({
        split: Dataset.from_list(_parse_iob_lines(list(raw[split])))
        for split in ("train", "validation", "test")
    })


def compute_class_weights(train_dataset: Dataset, cap: float = 10.0) -> list[float]:
    counts: collections.Counter = collections.Counter()
    for ex in train_dataset:
        for tag_id in ex["ner_tags"]:
            counts[tag_id] += 1
    total = sum(counts.values())
    n = len(LABEL_NAMES)
    raw_w = [1.0 / ((counts.get(i, 1) / total) * n) for i in range(n)]
    min_w = min(raw_w)
    return [min(w, min_w * cap) for w in raw_w]

In [ ]:
%%writefile /content/climate_ner/train.py
"""
train.py -- Fine-tune nasa-impact/nasa-smd-ibm-v0.1 (INDUS / RoBERTa-base)
            for token classification on ibm-research/Climate-Change-NER.

Import order is deliberate:
  - 'import evaluate as hf_evaluate' must come AFTER sys.path is clean.
  - Our evaluation module is named ner_eval.py (not evaluate.py) to avoid
    shadowing the installed HuggingFace 'evaluate' package.
"""
from __future__ import annotations
import argparse
import sys

# ── sys.path setup FIRST so our package is importable ────────────────────
sys.path.insert(0, '/content')

import numpy as np
import torch
import evaluate as hf_evaluate          # HuggingFace evaluate library
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)
from climate_ner.dataset import (
    LABEL_NAMES, LABEL2ID, ID2LABEL,
    load_climate_ner, compute_class_weights,
)

BASE_MODEL = "nasa-impact/nasa-smd-ibm-v0.1"
MAX_LENGTH = 512


def tokenize_and_align_labels(examples: dict, tokenizer) -> dict:
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    all_labels: list[list[int]] = []
    for i, word_labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        aligned, prev = [], None
        for wid in word_ids:
            if wid is None:
                aligned.append(-100)
            elif wid != prev:
                aligned.append(word_labels[wid])
            else:
                aligned.append(-100)
            prev = wid
        all_labels.append(aligned)
    tokenized["labels"] = all_labels
    return tokenized


class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        if class_weights is not None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
            self._cw = torch.tensor(class_weights, dtype=torch.float32, device=device)
        else:
            self._cw = None

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(weight=self._cw, ignore_index=-100)(
            outputs.logits.view(-1, model.config.num_labels), labels.view(-1)
        )
        return (loss, outputs) if return_outputs else loss


def build_compute_metrics(label_names):
    seqeval = hf_evaluate.load("seqeval")
    def compute_metrics(eval_pred):
        raw_preds, raw_labels = eval_pred
        pred_ids = np.argmax(raw_preds, axis=2)
        true_seqs, pred_seqs = [], []
        for pred_seq, label_seq in zip(pred_ids, raw_labels):
            tl, pl = [], []
            for p, l in zip(pred_seq, label_seq):
                if l == -100:
                    continue
                tl.append(label_names[l])
                pl.append(label_names[p])
            true_seqs.append(tl)
            pred_seqs.append(pl)
        r = seqeval.compute(predictions=pred_seqs, references=true_seqs)
        return {"precision": r["overall_precision"], "recall": r["overall_recall"],
                "f1": r["overall_f1"], "accuracy": r["overall_accuracy"]}
    return compute_metrics


def main(args):
    use_gpu = torch.cuda.is_available()
    print(f"\n{'='*60}")
    print(f"  Climate NER Fine-Tuning")
    print(f"  Base model : {BASE_MODEL}")
    print(f"  Device     : {'GPU (CUDA)' if use_gpu else 'CPU (slow!)'}") 
    print(f"  Output dir : {args.output_dir}")
    print(f"  Epochs={args.epochs}  LR={args.lr}  Batch={args.batch_size}")
    print(f"{'='*60}\n")

    print("-> [1/5] Loading dataset...")
    ds = load_climate_ner()
    print(f"   train={len(ds['train'])} | val={len(ds['validation'])} | test={len(ds['test'])}")

    print("-> [2/5] Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, add_prefix_space=True)

    print("-> [3/5] Tokenizing (subword alignment)...")
    tokenized_ds = ds.map(
        lambda batch: tokenize_and_align_labels(batch, tokenizer),
        batched=True, remove_columns=["tokens", "ner_tags"], desc="Tokenizing",
    )

    print("-> [4/5] Loading model + classification head...")
    model = AutoModelForTokenClassification.from_pretrained(
        BASE_MODEL, num_labels=len(LABEL_NAMES),
        id2label=ID2LABEL, label2id=LABEL2ID, ignore_mismatched_sizes=True,
    )

    print("-> [5/5] Using standard unweighted loss...")
    cw = None

    training_args = TrainingArguments(
        output_dir=args.output_dir,
        eval_strategy="epoch", save_strategy="epoch",
        learning_rate=args.lr, weight_decay=0.01,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size * 2,
        num_train_epochs=args.epochs,
        load_best_model_at_end=True, metric_for_best_model="f1", greater_is_better=True,
        fp16=use_gpu, logging_steps=25, push_to_hub=False, report_to="none",
    )

    trainer = WeightedLossTrainer(
        model=model, args=training_args,
        train_dataset=tokenized_ds["train"], eval_dataset=tokenized_ds["validation"],
        processing_class=tokenizer, data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=build_compute_metrics(LABEL_NAMES), class_weights=cw,
    )

    print("\n--- Training ---")
    result = trainer.train()
    print(f"\n   Runtime : {result.metrics.get('train_runtime', 0):.1f}s")
    print(f"   Loss    : {result.metrics.get('train_loss', 0):.4f}")

    print(f"\n-> Saving best model to: {args.output_dir}")
    trainer.save_model(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)
    print("checkmark Done. Run ner_eval.py next.")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--output-dir", default="/content/climate-ner-indus")
    parser.add_argument("--lr", type=float, default=2e-5)
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--epochs", type=int, default=5)
    main(parser.parse_args())

In [ ]:
%%writefile /content/climate_ner/ner_eval.py
"""
ner_eval.py -- Per-entity-type seqeval report on the test set.

Named ner_eval.py (NOT evaluate.py) to avoid shadowing the installed
HuggingFace 'evaluate' library when Python searches the script's directory.
"""
from __future__ import annotations
import argparse
import sys

sys.path.insert(0, '/content')

import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
from seqeval.metrics import classification_report, f1_score
from climate_ner.dataset import load_climate_ner, LABEL_NAMES


def evaluate_test_set(model_dir: str) -> None:
    device = 0 if torch.cuda.is_available() else -1
    print(f"\n-> Loading model from: {model_dir}")
    tokenizer = AutoTokenizer.from_pretrained(model_dir, add_prefix_space=True)
    model = AutoModelForTokenClassification.from_pretrained(model_dir)
    ner_pipe = pipeline("ner", model=model, tokenizer=tokenizer,
                        aggregation_strategy="none", device=device)

    print("-> Loading test split...")
    test_ds = load_climate_ner()["test"]
    print(f"   {len(test_ds)} sentences")

    true_seqs, pred_seqs = [], []
    for example in test_ds:
        tokens = example["tokens"]
        gold = [LABEL_NAMES[i] for i in example["ner_tags"]]
        text = " ".join(tokens)
        ner_output = ner_pipe(text)
        pred = ["O"] * len(tokens)
        seen = set()
        for ent in ner_output:
            word_idx = len(text[:ent.get("start", 0)].split())
            if word_idx < len(pred) and word_idx not in seen:
                pred[word_idx] = ent["entity"]
                seen.add(word_idx)
        true_seqs.append(gold)
        pred_seqs.append(pred)

    print("\n" + "=" * 70)
    print("  Per-Entity-Type Report (Test Set)")
    print("=" * 70)
    print(classification_report(true_seqs, pred_seqs, digits=4))
    print(f"  Overall seqeval micro-F1: {f1_score(true_seqs, pred_seqs):.4f}")
    print("=" * 70)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model-dir", default="/content/climate-ner-indus")
    evaluate_test_set(parser.parse_args().model_dir)

In [ ]:
%%writefile /content/climate_ner/predict.py
"""
predict.py -- ClimateNER inference wrapper.
"""
from __future__ import annotations
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline


class ClimateNER:
    def __init__(self, model_dir: str = "/content/climate-ner-indus") -> None:
        device = 0 if torch.cuda.is_available() else -1
        tokenizer = AutoTokenizer.from_pretrained(model_dir, add_prefix_space=True)
        model = AutoModelForTokenClassification.from_pretrained(model_dir)
        self._pipe = pipeline("ner", model=model, tokenizer=tokenizer,
                              aggregation_strategy="simple", device=device)

    def extract(self, text: str) -> list[dict]:
        return [{
            "entity_group": e["entity_group"].lstrip("BI-"),
            "word":  e["word"],
            "score": round(float(e["score"]), 4),
            "start": e["start"],
            "end":   e["end"],
        } for e in self._pipe(text)]

In [ ]:
# Confirm all files written
import os
files = os.listdir('/content/climate_ner')
print('Files in /content/climate_ner:', sorted(files))
assert 'dataset.py' in files
assert 'train.py'   in files
assert 'ner_eval.py' in files   # NOT evaluate.py
assert 'predict.py' in files
print('checkmark All source files present')

---
## 🔍 Cell 3 — Smoke-test the parser

In [ ]:
import sys
sys.path.insert(0, '/content')

from climate_ner.dataset import load_climate_ner, LABEL_NAMES, compute_class_weights

ds = load_climate_ner()
print(f'Train : {len(ds["train"])} sentences')
print(f'Val   : {len(ds["validation"])} sentences')
print(f'Test  : {len(ds["test"])} sentences')

ex = ds['train'][11]
print(f'\nSample tokens : {ex["tokens"][:7]}')
print(f'Sample labels : {[LABEL_NAMES[i] for i in ex["ner_tags"][:7]]}')

weights = compute_class_weights(ds['train'])
print(f'\nClass weights  : O={weights[0]:.4f}  max={max(weights):.4f}  (10x cap)')

assert all(0 <= t < len(LABEL_NAMES) for s in ds['train'] for t in s['ner_tags'])
assert len(ds['train']) >= 100
print('\n✓ Smoke-test passed — ready to train')

---
## ⚙️ Cell 4 — Configure hyperparameters
**Edit these before running Cell 5.**

In [ ]:
OUTPUT_DIR    = '/content/climate-ner-indus'
LEARNING_RATE = 2e-5
BATCH_SIZE    = 8     # reduce to 8 if you hit OOM
NUM_EPOCHS    = 8

print(f'Output dir  : {OUTPUT_DIR}')
print(f'LR          : {LEARNING_RATE}')
print(f'Batch size  : {BATCH_SIZE}')
print(f'Epochs      : {NUM_EPOCHS}')

---
## 🏋️ Cell 5 — Train
~12–18 min on T4 GPU.

In [ ]:
!python /content/climate_ner/train.py \
    --output-dir {OUTPUT_DIR} \
    --lr {LEARNING_RATE} \
    --batch-size {BATCH_SIZE} \
    --epochs {NUM_EPOCHS}

---
## 📊 Cell 6 — Evaluate on test set

In [ ]:
!python /content/climate_ner/ner_eval.py --model-dir {OUTPUT_DIR}

---
## 🔬 Cell 7 — Inference demo

In [ ]:
import sys
sys.path.insert(0, '/content')
from climate_ner.predict import ClimateNER

ner = ClimateNER(OUTPUT_DIR)

sentences = [
    "Intense El Niño rainfall triggered flash floods in lower regions, "
    "damaging maize crops and washing out local bridges.",

    "CO2 emissions from the transport sector accelerated permafrost thaw "
    "and sea-level rise in coastal communities.",

    "CMIP6 models project a 2°C increase in global mean surface temperature "
    "under RCP8.5 by 2100.",
]

for sent in sentences:
    print(f'\nInput: {sent}')
    entities = ner.extract(sent)
    if not entities:
        print('  (no entities)')
    else:
        for e in entities:
            print(f"  [{e['entity_group']:<30}] '{e['word']}' ({e['score']:.3f})")

---
## 💾 Cell 8 — (Optional) Save model to Google Drive

In [ ]:
# Uncomment to persist the trained model to your Drive after the session ends
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# DRIVE_PATH = '/content/drive/MyDrive/CLIMATE-TECH/models/climate-ner-indus'
# shutil.copytree(OUTPUT_DIR, DRIVE_PATH, dirs_exist_ok=True)
# print(f'Model saved to Drive: {DRIVE_PATH}')